# Day 7 Lab 05: Bronze Schema Drift

        **Layer:** Bronze  
        **Python reference:** `Lab_Files/labs/lab_05_bronze_schema_drift.py`

        This notebook is a sectioned companion version of the existing Python lab. It does not replace `run_lab.py` or the scripts under `Lab_Files/labs`.

        ## Learning Checkpoints
        - Read the Bronze order table.
- Profile every column for type and null counts.
- Use sparse optional columns as schema-drift signals.

**Dependency:** Run Lab/Notebook 04 first so `lake/bronze/orders_raw` exists.

## 0. Notebook Setup

Run this first. It moves the kernel into `Lab_Files` and makes the shared lab helpers importable.

In [ ]:
from pathlib import Path
import os
import sys
import types
import typing

# PySpark 3.4 imports typing.io, which is absent in Python 3.14+.
if "typing.io" not in sys.modules:
    typing_io = types.ModuleType("typing.io")
    typing_io.IO = typing.IO
    typing_io.TextIO = typing.TextIO
    typing_io.BinaryIO = typing.BinaryIO
    sys.modules["typing.io"] = typing_io

def find_lab_files(start: Path) -> Path:
    relative_options = [
        Path("."),
        Path("Lab_Files"),
        Path("Day_07") / "Lab_Files",
        Path("Week_02") / "Day_07" / "Lab_Files",
    ]
    for root in [start, *start.parents]:
        for relative in relative_options:
            candidate = (root / relative).resolve()
            if (candidate / "labs" / "day7_common.py").exists():
                return candidate
    raise FileNotFoundError(
        "Could not find Week_02/Day_07/Lab_Files. "
        "Start Jupyter from the repository root, Week_02/Day_07, or Week_02/Day_07/Lab_Files."
    )

HERE = Path.cwd().resolve()
LAB_FILES = find_lab_files(HERE)

os.chdir(LAB_FILES)
labs_path = str(LAB_FILES / "labs")
if labs_path not in sys.path:
    sys.path.insert(0, labs_path)

print(f"Working directory: {Path.cwd()}")
print(f"Using lab helpers from: {labs_path}")


## 1. Import Lab Helpers

In [ ]:
from pyspark.sql import functions as F

import importlib
import day7_common
day7_common = importlib.reload(day7_common)


from day7_common import OUTPUT_DIR, ensure_output_dirs, read_bronze_orders, require_source_data, spark_session, write_csv_dir

## 2. Start Spark and Read Bronze

In [ ]:
require_source_data()
ensure_output_dirs()
spark = spark_session("Day7Notebook05BronzeSchemaDrift")

bronze = read_bronze_orders(spark)
print("Bronze table loaded for profiling.")

## 3. Inspect Bronze Schema

In [ ]:
bronze.printSchema()
bronze.show(10, truncate=False)

## 4. Build Column Profile Rows

In [ ]:
metrics = bronze.agg(
    F.count(F.lit(1)).alias("__total_rows"),
    *[F.count(F.col(field.name)).alias(field.name) for field in bronze.schema.fields],
).first().asDict()
total_rows = metrics["__total_rows"]

profile_rows = []
for field in bronze.schema.fields:
    column = field.name
    non_null = metrics[column]
    profile_rows.append((column, field.dataType.simpleString(), non_null, total_rows - non_null))

print(f"Profiled {len(profile_rows)} columns with one Spark aggregation action.")

## 5. Create and Display Schema Profile

In [ ]:
profile = spark.createDataFrame(profile_rows, ["column_name", "data_type", "non_null_rows", "null_rows"])
profile.orderBy("column_name").show(50, truncate=False)

## 6. Write Profile Output

In [ ]:
write_csv_dir(profile.orderBy("column_name"), OUTPUT_DIR / "notebook_05_bronze_schema_profile")
print(f"Profiled {len(profile_rows)} Bronze columns across {total_rows} rows.")

## Clean Shutdown

Stop the SparkSession when you are done with the notebook. The next notebook will create its own session.

In [ ]:
# Run this at the end of the notebook, or before restarting/rerunning the lab.
spark.stop()